# Why tidypy

This notebook compares pure pandas and tidypy on the same cleaning task, focusing on readability, change surface, and identical results.


## What this example covers

1. Build a small employee dataset
2. Clean it with pure pandas
3. Clean the same data with tidypy
4. Compare grouped summaries
5. See which version is easier to extend


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from pandas.testing import assert_frame_equal


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    return start


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tidypy.tidy import (
    add_count,
    arrange,
    case_when,
    count,
    desc,
    filter_rows,
    glimpse,
    mutate_across,
    rename_with,
    select,
    starts_with,
    str_trim,
    summarize,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## 1. Build a small but realistic dataset

This dataset intentionally includes a few very common cleaning issues:

- missing numeric values
- padded strings
- repeated group values
- multiple score columns that need the same rule


In [2]:
df = pd.DataFrame(
    {
        "employee_id": [101, 102, 103, 104, 105, 106],
        "dept": ["Sales", "Sales", "Tech", "Tech", "Tech", "HR"],
        "city": ["Shanghai", "Shanghai", "Beijing", "Beijing", "Shenzhen", "Shanghai"],
        "score_math": [92.0, None, 88.0, 79.0, 95.0, None],
        "score_eng": [85.0, 90.0, None, 82.0, 91.0, 87.0],
        "name": [" Alice ", "Bob ", " Carol", "David", " Emma ", " Frank "],
    }
)

df


,employee_id,dept,city,score_math,score_eng,name
0,101,Sales,Shanghai,92.0,85.0,Alice
1,102,Sales,Shanghai,NaN,90.0,Bob
2,103,Tech,Beijing,88.0,NaN,Carol
3,104,Tech,Beijing,79.0,82.0,David
4,105,Tech,Shenzhen,95.0,91.0,Emma
5,106,HR,Shanghai,NaN,87.0,Frank


## 2. Inspect the table structure

Start with `glimpse()` before moving into the cleaning steps.


In [3]:
glimpse(df)


Rows: 6, Columns: 6


,column,dtype,non_null,nulls,n_unique,preview
0,employee_id,int64,6,0,6,"101, 102, 103"
1,dept,str,6,0,3,"'Sales', 'Sales', 'Tech'"
2,city,str,6,0,3,"'Shanghai', 'Shanghai', 'Beijing'"
3,score_math,float64,4,2,4,"92.0, NA, 88.0"
4,score_eng,float64,5,1,5,"85.0, 90.0, NA"
5,name,str,6,0,6,"' Alice ', 'Bob ', ' Carol'"


,column,dtype,non_null,nulls,n_unique,preview
0,employee_id,int64,6,0,6,"101, 102, 103"
1,dept,str,6,0,3,"'Sales', 'Sales', 'Tech'"
2,city,str,6,0,3,"'Shanghai', 'Shanghai', 'Beijing'"
3,score_math,float64,4,2,4,"92.0, NA, 88.0"
4,score_eng,float64,5,1,5,"85.0, 90.0, NA"
5,name,str,6,0,6,"' Alice ', 'Bob ', ' Carol'"


In [4]:
print(glimpse(df, cols=starts_with("score_"), as_text=True, display=False, width=2, max_width=12))


Rows: 6
Columns: 2
$ score_math <float64> [non-null=4, nulls=2, unique=4] 92.0, NA
$ score_eng <float64> [non-null=5, nulls=1, unique=5] 85.0, 90.0


## 3. Pure pandas version

This block writes the full cleaning flow directly in pandas as the baseline.


In [5]:
pandas_result = (
    df.loc[:, ["employee_id", "dept", "city", "score_math", "score_eng", "name"]]
    .assign(
        score_math=lambda x: x["score_math"].fillna(x["score_math"].median()),
        score_eng=lambda x: x["score_eng"].fillna(x["score_eng"].median()),
    )
    .rename(columns={"score_math": "math", "score_eng": "eng"})
    .assign(
        name=lambda x: x["name"].str.strip(),
        level=lambda x: pd.Series(
            ["A" if v >= 90 else "B" if v >= 80 else "C" for v in x["math"]],
            index=x.index,
        ),
        dept_size=lambda x: x.groupby("dept")["employee_id"].transform("size"),
    )
    .loc[lambda x: x["name"].str.contains(r"^[ACDEF]", regex=True, na=False)]
    .sort_values(by=["dept", "math"], ascending=[True, False])
    .reset_index(drop=True)
)

pandas_result


,employee_id,dept,city,math,eng,name,level,dept_size
0,106,HR,Shanghai,90.0,87.0,Frank,A,1
1,101,Sales,Shanghai,92.0,85.0,Alice,A,2
2,105,Tech,Shenzhen,95.0,91.0,Emma,A,3
3,103,Tech,Beijing,88.0,87.0,Carol,B,3
4,104,Tech,Beijing,79.0,82.0,David,C,3


## 4. tidypy version

The logic stays the same. Only the expression style changes.


In [6]:
tidypy_result = (
    df
    .pipe(
        select,
        "employee_id",
        "dept",
        "city",
        starts_with("score_"),
        "name",
    )
    .pipe(
        mutate_across,
        starts_with("score_"),
        lambda s: s.fillna(s.median()),
    )
    .pipe(
        rename_with,
        lambda c: c.replace("score_", ""),
        starts_with("score_"),
    )
    .assign(
        name=lambda x: str_trim(x["name"]),
        level=lambda x: case_when(
            (x["math"] >= 90, "A"),
            (x["math"] >= 80, "B"),
            default="C",
        ),
    )
    .pipe(
        add_count,
        "dept",
        name="dept_size",
    )
    .pipe(
        filter_rows,
        lambda x: x["name"].str.contains(r"^[ACDEF]", regex=True, na=False),
    )
    .pipe(
        arrange,
        "dept",
        desc("math"),
    )
    .reset_index(drop=True)
)

tidypy_result


,employee_id,dept,city,math,eng,name,level,dept_size
0,106,HR,Shanghai,90.0,87.0,Frank,A,1
1,101,Sales,Shanghai,92.0,85.0,Alice,A,2
2,105,Tech,Shenzhen,95.0,91.0,Emma,A,3
3,103,Tech,Beijing,88.0,87.0,Carol,B,3
4,104,Tech,Beijing,79.0,82.0,David,C,3


In [7]:
assert_frame_equal(pandas_result, tidypy_result)
print("The pure pandas result and the tidypy result are identical.")


The pure pandas result and the tidypy result are identical.


## 5. Grouped summaries

Once the cleaned results match, compare the summary step as well.


In [8]:
pandas_summary = (
    tidypy_result.groupby(["dept", "city"], sort=False)
    .agg(
        avg_math=("math", "mean"),
        avg_eng=("eng", "mean"),
        n=("employee_id", "count"),
    )
    .reset_index()
)

pandas_summary


,dept,city,avg_math,avg_eng,n
0,HR,Shanghai,90.0,87.0,1
1,Sales,Shanghai,92.0,85.0,1
2,Tech,Shenzhen,95.0,91.0,1
3,Tech,Beijing,83.5,84.5,2


In [9]:
tidypy_summary = summarize(
    tidypy_result,
    by=["dept", "city"],
    avg_math=("math", "mean"),
    avg_eng=("eng", "mean"),
    n=("employee_id", "count"),
)

tidypy_summary


,dept,city,avg_math,avg_eng,n
0,HR,Shanghai,90.0,87.0,1
1,Sales,Shanghai,92.0,85.0,1
2,Tech,Shenzhen,95.0,91.0,1
3,Tech,Beijing,83.5,84.5,2


In [10]:
assert_frame_equal(pandas_summary, tidypy_summary)
count(tidypy_result, "dept")


,dept,n
0,HR,1
1,Sales,1
2,Tech,3


## 6. What tidypy is really improving here

In this example, tidypy is not changing the result. It is mostly improving how the code is organized:

- column families are more explicit
- one rule can be applied to a whole group of columns more naturally
- bulk renaming avoids repeating column names
- adding a new column later usually touches fewer lines


## Exercise

Add a `score_logic` column, then update both the pandas version and the tidypy version so that they:

1. include it in the cleaning flow
2. fill missing values with the median
3. include `avg_logic` in the final summary

After that, compare how many lines changed on each side.


In [11]:
# Answer scaffold
# 1. Add a score_logic column to df.
# 2. Re-run the pure pandas block.
# 3. Re-run the tidypy block.
# 4. Compare how many lines changed.
